In [1]:
import yfinance as yf
import numpy as np
import pickle
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

In [2]:
def train_and_save_model(ticker):
    print(f"Training model for {ticker}...")
    
    # Fetch data
    stock = yf.Ticker(ticker)
    data = stock.history(period="1y")
    data.index = data.index.tz_localize(None)
    closes = data["Close"].values.reshape(-1, 1)

    # Scale data
    scaler = MinMaxScaler(feature_range=(0, 1))
    scaled = scaler.fit_transform(closes)

    # Create sequences
    sequence_length = 60
    X, y = [], []
    for i in range(sequence_length, len(scaled)):
        X.append(scaled[i-sequence_length:i, 0])
        y.append(scaled[i, 0])

    X, y = np.array(X), np.array(y)
    X = X.reshape(X.shape[0], X.shape[1], 1)

    # Split
    split = int(len(X) * 0.8)
    X_train = X[:split]
    y_train = y[:split]

    # Build model
    model = Sequential([
        LSTM(32, return_sequences=False, input_shape=(sequence_length, 1)),
        Dropout(0.2),
        Dense(16),
        Dense(1)
    ])
    model.compile(optimizer="adam", loss="mean_squared_error")

    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True
    )

    model.fit(
        X_train, y_train,
        epochs=20,
        batch_size=32,
        validation_split=0.1,
        callbacks=[early_stop],
        verbose=1
    )

    # Save model + scaler together in pkl
    model_data = {
        "scaler": scaler,
        "weights": model.get_weights(),
        "sequence_length": sequence_length
    }

    ticker_clean = ticker.replace(".", "_")
    filename = f"model_{ticker_clean}.pkl"
    with open(filename, "wb") as f:
        pickle.dump(model_data, f)

    print(f"✅ Model saved as {filename}")
    return filename

print("Function ready!")

Function ready!
